In [7]:
import os
import glob
import html
import warnings

import numpy as np
import pandas as pd
import joblib
from IPython.display import display, HTML, clear_output

warnings.filterwarnings("ignore")


def _ensure(pkg, import_name=None):
    """Import a package, pip-installing it first if Kaggle doesn't have it."""
    import importlib
    import subprocess
    import sys
    try:
        return importlib.import_module(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        return importlib.import_module(import_name or pkg)


folium = _ensure("folium")


In [8]:
DATA_PATH = "/kaggle/input/datasets/abinashnayak11/hospital"                                                                          
FEE_WEIGHT = 1.0
DISTANCE_WEIGHT = 1.0
RATING_WEIGHT = 1000.0
                                                   
                                                             
RANK_FULL_HOSPITALS_LAST = True

                                      
LIMITED_OCCUPANCY_PCT = 90                                       
LIMITED_BEDS_LEFT = 5                                            

DEFAULT_TOP_N = 3

                                                                  
TRAIN_ML_CLASSIFIER = True
N_SYNTHETIC_USERS = 2000                                                    
MAX_TOP_N_FOR_TRAINING = 10
RANDOM_STATE = 42

OUTPUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."


In [9]:
REQUIRED_COLS = [
    "hospital_id", "hospital_name", "hospital_type", "address",
    "latitude", "longitude",
    "general_ward_total_beds", "general_ward_available_beds",
    "general_ward_occupancy_pct", "admission_fee_inr",
    "rating_5", "review_count",
]
NUMERIC_COLS = [
    "latitude", "longitude", "general_ward_total_beds",
    "general_ward_available_beds", "general_ward_occupancy_pct",
    "admission_fee_inr", "rating_5", "review_count",
]
WEBSITE_ALIASES = ["website", "website_url", "hospital_website", "url", "link", "web"]


def find_csv(path):
    """Use `path` if it exists; otherwise look for a matching CSV in /kaggle/input."""
    if path and os.path.isfile(path):
        return path
    for cand in sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True)):
        try:
            cols = {str(c).strip().lower() for c in pd.read_csv(cand, nrows=0).columns}
        except Exception:
            continue
        if set(REQUIRED_COLS) <= cols:
            print(f"DATA_PATH not found - using detected file: {cand}")
            return cand
    raise FileNotFoundError(
        f"Could not find your CSV at DATA_PATH = {path!r}.\n"
        "Add the dataset to the notebook and set DATA_PATH in Block 2 "
        "(copy the path from the right-hand 'Input' panel)."
    )


def load_hospitals_csv(path=None):
    return pd.read_csv(find_csv(path or DATA_PATH))


def prepare_hospitals(raw):
    """Validate + clean the raw dataframe. Returns a new dataframe."""
    df = raw.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]

    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"CSV is missing required columns: {missing}")

    for c in NUMERIC_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")

                                                         
    df = df.dropna(subset=["latitude", "longitude"])
    df = df[df["latitude"].between(-90, 90) & df["longitude"].between(-180, 180)]
    df = df.drop_duplicates(subset="hospital_id").reset_index(drop=True)
    if df.empty:
        raise ValueError("No hospitals with valid latitude/longitude found in the CSV.")

          
    avail = df["general_ward_available_beds"].fillna(0).clip(lower=0)
    total = df["general_ward_total_beds"].fillna(0).clip(lower=0)
    total = np.maximum(total, avail)                                              
    df["general_ward_available_beds"] = avail.astype(int)
    df["general_ward_total_beds"] = total.astype(int)

    occ_calc = pd.Series(
        np.where(total > 0, (total - avail) / total.replace(0, np.nan) * 100, 100.0),
        index=df.index,
    )
    df["general_ward_occupancy_pct"] = (
        df["general_ward_occupancy_pct"].fillna(occ_calc).clip(0, 100)
    )

                            
    df["admission_fee_inr"] = (
        df["admission_fee_inr"].fillna(df["admission_fee_inr"].median()).fillna(0).clip(lower=0)
    )
    df["rating_5"] = df["rating_5"].fillna(df["rating_5"].median()).fillna(0).clip(0, 5)
    df["review_count"] = df["review_count"].fillna(0).clip(lower=0).astype(int)

                  
    df["hospital_name"] = df["hospital_name"].fillna("Unnamed hospital").astype(str)
    df["hospital_type"] = df["hospital_type"].fillna("Unknown").astype(str)
    df["address"] = df["address"].fillna("").astype(str)

                             
    site_col = next((c for c in WEBSITE_ALIASES if c in df.columns), None)
    if site_col:
        site = df[site_col].fillna("").astype(str).str.strip()
        site = site.where(site.str.lower().str.startswith(("http://", "https://")) | (site == ""),
                          "https://" + site)
        df["website"] = site
    else:
        df["website"] = ""
    return df


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle (straight-line) distance in km. Works with numpy arrays."""
    R = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = p2 - p1
    dlmb = np.radians(lon2) - np.radians(lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlmb / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


In [10]:
class HospitalModel:
    ML_FEATURES = [
        "distance_km", "admission_fee_inr", "rating_5", "review_count",
        "general_ward_available_beds", "general_ward_occupancy_pct",
        "distance_rank", "fee_rank", "rating_rank", "top_n",
    ]

    def __init__(self, fee_weight=FEE_WEIGHT, distance_weight=DISTANCE_WEIGHT,
                 rating_weight=RATING_WEIGHT, rank_full_last=RANK_FULL_HOSPITALS_LAST,
                 limited_occupancy_pct=LIMITED_OCCUPANCY_PCT,
                 limited_beds_left=LIMITED_BEDS_LEFT, random_state=RANDOM_STATE):
        self.fee_weight = fee_weight
        self.distance_weight = distance_weight
        self.rating_weight = rating_weight
        self.rank_full_last = rank_full_last
        self.limited_occupancy_pct = limited_occupancy_pct
        self.limited_beds_left = limited_beds_left
        self.random_state = random_state
        self.hospitals = None
        self.classifier = None
        self.metrics_ = None

                                                                            
    def fit(self, raw_df, train_classifier=TRAIN_ML_CLASSIFIER):
        self.hospitals = prepare_hospitals(raw_df)
        print(f"Loaded {len(self.hospitals)} hospitals.")
        if train_classifier:
            self._train_classifier()
        return self

    @property
    def has_website(self):
        return bool((self.hospitals["website"] != "").any())

                                                                              
    def _score(self, user_lat, user_lon):
        d = self.hospitals.copy()
        d["distance_km"] = haversine_km(user_lat, user_lon,
                                        d["latitude"].values, d["longitude"].values)
        d["fee_component"] = -self.fee_weight * d["admission_fee_inr"]                                    
        d["distance_component"] = -self.distance_weight * d["distance_km"]                                     
        d["rating_component"] = self.rating_weight * d["rating_5"]                                            
        d["score"] = d["fee_component"] + d["distance_component"] + d["rating_component"]
        return d

    def _bed_status(self, d):
        avail = d["general_ward_available_beds"]
        occ = d["general_ward_occupancy_pct"]
        return np.select(
            [avail <= 0, (occ >= self.limited_occupancy_pct) | (avail <= self.limited_beds_left)],
            ["Full", "Limited"],
            default="Available",
        )

    def rank(self, user_lat, user_lon, top_n=DEFAULT_TOP_N):
        """Return ALL hospitals, ranked best-first, with distance, score and labels."""
        if not (-90 <= user_lat <= 90 and -180 <= user_lon <= 180):
            raise ValueError("Latitude must be in [-90, 90] and longitude in [-180, 180].")
        n = len(self.hospitals)
        top_n = int(np.clip(top_n, 1, n))

        d = self._score(user_lat, user_lon)
        d["has_beds"] = d["general_ward_available_beds"] > 0
        if self.rank_full_last:
            d = d.sort_values(["has_beds", "score", "distance_km"], ascending=[False, False, True])
        else:
            d = d.sort_values(["score", "distance_km"], ascending=[False, True])
        d = d.reset_index(drop=True)

        d["rank"] = np.arange(1, n + 1)
        d["recommended"] = d["rank"] <= top_n
        if self.rank_full_last:
            d["recommended"] = d["recommended"] & d["has_beds"]
        d["recommendation"] = np.where(d["recommended"], "Recommended", "Other")
        d["bed_status"] = self._bed_status(d)
        return d

    def recommend(self, user_lat, user_lon, top_n=DEFAULT_TOP_N):
        """JSON-friendly output for a web / mobile backend."""
        d = self.rank(user_lat, user_lon, top_n)
        cols = ["rank", "hospital_id", "hospital_name", "hospital_type", "address",
                "latitude", "longitude", "distance_km", "admission_fee_inr", "rating_5",
                "review_count", "general_ward_available_beds", "general_ward_total_beds",
                "general_ward_occupancy_pct", "bed_status", "score", "recommendation", "website"]
        out = d[cols].copy()
        out["distance_km"] = out["distance_km"].round(2)
        out["score"] = out["score"].round(2)
        return out.to_dict("records")

                                                                              
    def _ml_features(self, d, top_n):
        return pd.DataFrame({
            "distance_km": d["distance_km"].values,
            "admission_fee_inr": d["admission_fee_inr"].values,
            "rating_5": d["rating_5"].values,
            "review_count": d["review_count"].values,
            "general_ward_available_beds": d["general_ward_available_beds"].values,
            "general_ward_occupancy_pct": d["general_ward_occupancy_pct"].values,
            "distance_rank": d["distance_km"].rank(method="min").values,
            "fee_rank": d["admission_fee_inr"].rank(method="min").values,
            "rating_rank": d["rating_5"].rank(ascending=False, method="min").values,
            "top_n": top_n,
        })[self.ML_FEATURES]

    def _train_classifier(self, n_users=N_SYNTHETIC_USERS):
        """
        Learn 'Recommended vs Other' from simulated users.
        Because a hospital's label depends on where the user is, we simulate
        many random user locations (and Top-N values) around the hospitals,
        label each hospital with the scoring rule, and train a RandomForest.
        Evaluation uses a hold-out of unseen user locations.
        """
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
        from sklearn.model_selection import GroupShuffleSplit

        rng = np.random.default_rng(self.random_state)
        h = self.hospitals
        n = len(h)
        max_n = max(1, min(MAX_TOP_N_FOR_TRAINING, n))

        lat_min, lat_max = h["latitude"].min(), h["latitude"].max()
        lon_min, lon_max = h["longitude"].min(), h["longitude"].max()
        pad_lat = max((lat_max - lat_min) * 0.15, 0.02)
        pad_lon = max((lon_max - lon_min) * 0.15, 0.02)

        Xs, ys, groups = [], [], []
        for u in range(n_users):
            lat = rng.uniform(max(-90, lat_min - pad_lat), min(90, lat_max + pad_lat))
            lon = rng.uniform(max(-180, lon_min - pad_lon), min(180, lon_max + pad_lon))
            top_n = int(rng.integers(1, max_n + 1))
            d = self.rank(lat, lon, top_n)
            Xs.append(self._ml_features(d, top_n))
            ys.append(d["recommended"].astype(int).values)
            groups.append(np.full(n, u))

        X = pd.concat(Xs, ignore_index=True)
        y = np.concatenate(ys)
        g = np.concatenate(groups)

        if len(np.unique(y)) < 2:
            print("ML classifier skipped: training labels contain only one class.")
            return

        def new_clf():
          return RandomForestClassifier( n_estimators=200,max_depth=4,
                                         min_samples_split=40,min_samples_leaf=20,
                                             max_features=0.7,class_weight="balanced",
                                            n_jobs=-1,random_state=self.random_state)
        tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25,
                                        random_state=self.random_state).split(X, y, g))
        clf = new_clf().fit(X.iloc[tr], y[tr])
        pred = clf.predict(X.iloc[te])

        self.metrics_ = {
            "accuracy": accuracy_score(y[te], pred),
            "confusion_matrix": confusion_matrix(y[te], pred),
        }
        print("\n=== ML classifier: Recommended vs Other (hold-out user locations) ===")
        print(f"Accuracy: {self.metrics_['accuracy']:.4f}")
        print(classification_report(y[te], pred, target_names=["Other", "Recommended"], digits=3))
        print("Confusion matrix [rows = actual, cols = predicted]:")
        print(self.metrics_["confusion_matrix"])

                                             
        self.classifier = new_clf().fit(X, y)
        imp = pd.Series(self.classifier.feature_importances_, index=self.ML_FEATURES)
        print("\nFeature importance:")
        print(imp.sort_values(ascending=False).round(3).to_string())

    def predict_ml(self, user_lat, user_lon, top_n=DEFAULT_TOP_N):
        """Ranked table plus the ML classifier's own Recommended / Other prediction."""
        if self.classifier is None:
            raise RuntimeError("ML classifier not trained (set TRAIN_ML_CLASSIFIER = True).")
        d = self.rank(user_lat, user_lon, top_n)
        d["ml_recommended"] = self.classifier.predict(self._ml_features(d, int(top_n))).astype(bool)
        return d

                                                                              
    def _config(self):
        return dict(fee_weight=self.fee_weight, distance_weight=self.distance_weight,
                    rating_weight=self.rating_weight, rank_full_last=self.rank_full_last,
                    limited_occupancy_pct=self.limited_occupancy_pct,
                    limited_beds_left=self.limited_beds_left, random_state=self.random_state)

    def save(self, path=None):
        path = path or os.path.join(OUTPUT_DIR, "hospital_model.joblib")
        joblib.dump({"config": self._config(), "hospitals": self.hospitals,
                     "classifier": self.classifier, "metrics": self.metrics_}, path)
        print(f"Model saved to {path}")
        return path

    @classmethod
    def load(cls, path):
        blob = joblib.load(path)
        m = cls(**blob["config"])
        m.hospitals, m.classifier, m.metrics_ = blob["hospitals"], blob["classifier"], blob["metrics"]
        return m


In [11]:
def directions_url(ulat, ulon, hlat, hlon):
    return (f"https://www.google.com/maps/dir/?api=1&origin={ulat},{ulon}"
            f"&destination={hlat},{hlon}&travelmode=driving")


def build_map(ranked, user_lat, user_lon):
    """User marker + ALL hospitals. Recommended (Top-N) ones show their rank number."""
    m = folium.Map(location=[user_lat, user_lon], zoom_start=12,
                   control_scale=True, width="100%", height=520)

    folium.Marker(
        [user_lat, user_lon], tooltip="You are here",
        icon=folium.Icon(color="blue", icon="user", prefix="fa"),
    ).add_to(m)

    for r in ranked.itertuples():
        name = html.escape(r.hospital_name)
        site = (f'<br><a href="{html.escape(r.website)}" target="_blank">Website</a>'
                if r.website else "")
        popup_html = (
            f"<b>#{r.rank} {name}</b><br>{html.escape(r.hospital_type)}<br>"
            f"{html.escape(r.address)}<br><hr style='margin:4px 0'>"
            f"Distance: {r.distance_km:.2f} km<br>"
            f"Admission fee: &#8377;{r.admission_fee_inr:,.0f}<br>"
            f"Rating: {r.rating_5:.1f}/5 ({r.review_count:,} reviews)<br>"
            f"General-ward beds: {r.general_ward_available_beds} free of {r.general_ward_total_beds} "
            f"({r.bed_status})<br>"
            f'<a href="{directions_url(user_lat, user_lon, r.latitude, r.longitude)}" '
            f'target="_blank">Get directions</a>{site}'
        )
        popup = folium.Popup(popup_html, max_width=300)

        if r.recommended:
            badge = (
                "<div style=\"background:#1b8a3a;color:#fff;border:2px solid #fff;"
                "border-radius:50%;width:30px;height:30px;line-height:26px;text-align:center;"
                f"font-weight:700;font-size:14px;box-shadow:0 0 4px rgba(0,0,0,.6)\">{r.rank}</div>"
            )
            folium.Marker(
                [r.latitude, r.longitude], popup=popup,
                tooltip=f"#{r.rank} {r.hospital_name}",
                icon=folium.DivIcon(html=badge, icon_size=(30, 30), icon_anchor=(15, 15)),
                z_index_offset=1000,
            ).add_to(m)
        else:
            colour = "#d32f2f" if r.bed_status == "Full" else "#757575"
            folium.CircleMarker(
                [r.latitude, r.longitude], radius=8, color="#ffffff", weight=2,
                fill=True, fill_color=colour, fill_opacity=0.95,
                popup=popup, tooltip=r.hospital_name,
            ).add_to(m)

    pts = ranked[["latitude", "longitude"]].values.tolist() + [[user_lat, user_lon]]
    m.fit_bounds([[min(p[0] for p in pts), min(p[1] for p in pts)],
                  [max(p[0] for p in pts), max(p[1] for p in pts)]], padding=(30, 30))
    return m


def build_table(ranked, user_lat, user_lon, has_website):
    """Table of ALL hospitals: rank, distance, fee, rating/reviews, website, etc."""
    d = ranked
    out = pd.DataFrame({
        "Rank": d["rank"].values,
        "Hospital": d["hospital_name"].map(html.escape).values,
        "Type": d["hospital_type"].map(html.escape).values,
        "Distance (km)": d["distance_km"].round(2).values,
        "Admission fee (INR)": d["admission_fee_inr"].map(lambda v: f"{v:,.0f}").values,
        "Rating (/5)": d["rating_5"].round(1).values,
        "Reviews": d["review_count"].map(lambda v: f"{int(v):,}").values,
        "Beds free / total": (d["general_ward_available_beds"].astype(str) + " / "
                              + d["general_ward_total_beds"].astype(str)).values,
        "Bed status": d["bed_status"].values,
        "Score": d["score"].round(1).values,
        "Recommended": np.where(d["recommended"], "★ Yes", "–"),
    })
    if has_website:
        out["Website"] = [f'<a href="{html.escape(u)}" target="_blank">Visit</a>' if u else "–"
                          for u in d["website"]]
    out["Directions"] = [
        f'<a href="{directions_url(user_lat, user_lon, la, lo)}" target="_blank">Route</a>'
        for la, lo in zip(d["latitude"], d["longitude"])
    ]

    def row_style(row):
        if str(row["Recommended"]).startswith("★"):
            css = "background-color:#d9f2e0;color:#111"
        elif row["Bed status"] == "Full":
            css = "background-color:#fde2e2;color:#111"
        else:
            css = ""
        return [css] * len(row)

    return out.style.hide(axis="index").apply(row_style, axis=1)


def show_results(model, user_lat, user_lon, top_n=DEFAULT_TOP_N, save_map=True):
    """Static (no widgets) version: map + table for one location."""
    ranked = model.rank(user_lat, user_lon, top_n)
    m = build_map(ranked, user_lat, user_lon)
    if save_map:
        path = os.path.join(OUTPUT_DIR, "hospital_map.html")
        m.save(path)
    display(m)
    display(HTML("<p><b>Green numbered circles</b> = Top-N recommended &nbsp;|&nbsp; "
                 "<b>grey</b> = other hospitals &nbsp;|&nbsp; <b>red</b> = no free General-Ward beds. "
                 "You can choose any hospital - rankings are only a suggestion.</p>"))
    display(build_table(ranked, user_lat, user_lon, model.has_website))
    return ranked


                                                                               
def _geocode(address):
    geopy_geocoders = _ensure("geopy", "geopy.geocoders")
    loc = geopy_geocoders.Nominatim(user_agent="hospital_model_kaggle").geocode(address, timeout=10)
    if loc is None:
        raise ValueError("Address not found - try a more specific address or enter lat/lon manually.")
    return float(loc.latitude), float(loc.longitude)


def _ip_location():
    """
    Approximate 'current location' from the IP address.
    NOTE: on Kaggle this is the location of Kaggle's server, not your device.
    In a real browser / mobile app use the device GPS (navigator.geolocation) and
    pass the coordinates to model.rank(lat, lon, top_n).
    """
    import requests
    sources = [
        ("https://ipapi.co/json/", lambda j: (j["latitude"], j["longitude"])),
        ("http://ip-api.com/json/", lambda j: (j["lat"], j["lon"])),
    ]
    for url, parse in sources:
        try:
            r = requests.get(url, timeout=8)
            r.raise_for_status()
            lat, lon = parse(r.json())
            return float(lat), float(lon)
        except Exception:
            continue
    raise RuntimeError("Could not detect location (turn Internet ON in Kaggle settings, "
                       "or enter latitude/longitude manually).")


def resolve_location(mode, lat, lon, address):
    if mode.startswith("Manual"):
        return float(lat), float(lon)
    if mode.startswith("Address"):
        if not address.strip():
            raise ValueError("Please type an address first.")
        return _geocode(address.strip())
    return _ip_location()


def launch_app(model):
    """Interactive app: location picker + Top-N slider + map + table."""
    import ipywidgets as w

    n = len(model.hospitals)
    state = {"lat": float(model.hospitals["latitude"].mean()),
             "lon": float(model.hospitals["longitude"].mean())}

    mode = w.Dropdown(
        options=["Manual (latitude / longitude)", "Address (needs Internet)",
                 "Current location (approx., IP-based)"],
        description="Location:", layout=w.Layout(width="460px"),
    )
    lat_in = w.FloatText(value=state["lat"], description="Latitude:", step=0.001)
    lon_in = w.FloatText(value=state["lon"], description="Longitude:", step=0.001)
    addr_in = w.Text(placeholder="Street / area, city", description="Address:",
                     layout=w.Layout(width="460px"))
    go = w.Button(description="Find hospitals", button_style="primary", icon="search")
    top_n = w.IntSlider(value=min(DEFAULT_TOP_N, n), min=1, max=n, step=1,
                        description="Top N:", continuous_update=False,
                        layout=w.Layout(width="460px"))
    msg = w.HTML()
    out = w.Output()

    def render():
        with out:
            clear_output(wait=True)
            ranked = model.rank(state["lat"], state["lon"], top_n.value)
            display(HTML(f"<p><b>Your location:</b> {state['lat']:.5f}, {state['lon']:.5f} "
                         f"&nbsp;|&nbsp; <b>Showing Top {top_n.value}</b> of {n} hospitals</p>"))
            display(build_map(ranked, state["lat"], state["lon"]))
            display(HTML("<p><b>Green numbered circles</b> = Top-N recommended &nbsp;|&nbsp; "
                         "<b>grey</b> = other hospitals &nbsp;|&nbsp; <b>red</b> = no free General-Ward beds. "
                         "Rankings are a suggestion - you can choose any hospital.</p>"))
            display(build_table(ranked, state["lat"], state["lon"], model.has_website))

    def on_go(_):
        try:
            lat, lon = resolve_location(mode.value, lat_in.value, lon_in.value, addr_in.value)
            state["lat"], state["lon"] = lat, lon
            lat_in.value, lon_in.value = lat, lon
            msg.value = ""
            render()
        except Exception as e:
            msg.value = f"<span style='color:#b00020'>&#9888; {html.escape(str(e))}</span>"

    go.on_click(on_go)
    top_n.observe(lambda change: render(), names="value")

    display(w.VBox([mode, lat_in, lon_in, addr_in, go, msg, top_n, out]))
    render()


In [12]:
raw_df = load_hospitals_csv(DATA_PATH)
model = HospitalModel().fit(raw_df)
model.save()

launch_app(model)


DATA_PATH not found - using detected file: /kaggle/input/datasets/abinashnayak11/hospital/hospitals.csv
Loaded 21 hospitals.

=== ML classifier: Recommended vs Other (hold-out user locations) ===
Accuracy: 0.9547
              precision    recall  f1-score   support

       Other      0.994     0.945     0.969      7758
 Recommended      0.863     0.983     0.919      2742

    accuracy                          0.955     10500
   macro avg      0.928     0.964     0.944     10500
weighted avg      0.959     0.955     0.956     10500

Confusion matrix [rows = actual, cols = predicted]:
[[7329  429]
 [  47 2695]]

Feature importance:
fee_rank                       0.311
admission_fee_inr              0.282
top_n                          0.163
review_count                   0.115
rating_5                       0.063
rating_rank                    0.042
general_ward_occupancy_pct     0.014
general_ward_available_beds    0.010
distance_km                    0.000
distance_rank              